In [1]:
#@title GioGuessr v1.0 [click 'Run all' to start]
#TODOs:
#filter by stars, epoch, team config, modifiers?, mapwidth height
#get a score based on how fast u answer
#display other info as hints? revealing affects ur score
  #hint:afks bc thats impossible to tell currently
  #hint:import simulator to create hints like first 25 turns or whatever
#ask user to guess winner distro for partial points
#score distro using loss func
#can use random seed or receive a fixed list of replayid so u can send to friend
#start from replayId list? (needs the ranking from l=bigteam API bc not in gior)
#can build model to guess. need good features first
CONST_SETTINGS = {'numTeams':4,'teamSize':3} #hard coded for most recent bigteam team config lol

In [2]:
#@title install (only need to run once) takes like 1-2 min lol
!apt-get install nodejs -y
!apt-get install npm -y


!npm install lz-string

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  javascript-common libc-ares2 libjs-highlight.js libnode72 nodejs-doc
Suggested packages:
  apache2 | lighttpd | httpd npm
The following NEW packages will be installed:
  javascript-common libc-ares2 libjs-highlight.js libnode72 nodejs nodejs-doc
0 upgraded, 6 newly installed, 0 to remove and 35 not upgraded.
Need to get 13.7 MB of archives.
After this operation, 54.0 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 javascript-common all 11+nmu1 [5,936 B]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libjs-highlight.js all 9.18.5+dfsg1-1 [367 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libc-ares2 amd64 1.18.1-1ubuntu0.22.04.3 [45.1 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 libnode72 amd64 12.22.9~dfsg-1ubuntu3.6 [10.8 MB]


In [3]:
#@title helper 1
import requests

def run_js_and_get_result(js_filename):
    # Run the JavaScript script with Node.js
    !node {js_filename} > decompressed_output.json

    # Read the decompressed JSON result
    with open("decompressed_output.json", "r") as f:
        decompressed_data = f.read()

    return decompressed_data


import requests
import json
import os

download_gior_async_hit_miss = [0,0] #hit/miss
async def download_gior_async(replay_id, session):
    if os.path.exists(f"/content/{replay_id}.gior"):
      download_gior_async_hit_miss[0]+=1
      return "gior in /content/ already"
    download_gior_async_hit_miss[1]+=1
    url = f"https://generalsio-replays-na.s3.amazonaws.com/{replay_id}.gior"
    async with session.get(url) as response:
        if response.status == 200:

            return await response.read()
            #return response.content  # Return the binary content directly
        else:
            print(f"Failed to fetch {replay_id}.gior: {response.status}")
            return None

# def download_gior(replay_id):
#     url = f"https://generalsio-replays-na.s3.amazonaws.com/{replay_id}.gior"
#     response = requests.get(url)
#     if response.status_code == 200:
#         return response.content  # Return the binary content directly
#     else:
#         print(f"Failed to fetch {replay_id}.gior: {response.status_code}")
#         return None


def create_js_script_async(binary_data, replay_id):
    # Convert binary data to a JavaScript array
    js_array = ','.join(map(str, binary_data))
    js_code = f"""
const LZString = require('lz-string');

// Decompress the data from the provided binary array
const uint8Array = new Uint8Array([{js_array}]);
const decompressed = LZString.decompressFromUint8Array(uint8Array);

// Output the decompressed JSON
if (decompressed) {{
    console.log(decompressed);
}} else {{
    console.log('Decompression failed.');
}}
    """
    with open('decompress_async.js', 'w') as file:
        file.write(js_code)
    return True


async def get_gior_data_async(replay_id, session):
    binary_data = await download_gior_async(replay_id, session)
    if binary_data:
        assert binary_data!="gior in /content/ already", 'bruhbruh'
        create_js_script_async(binary_data, replay_id)
        decompressed_data = run_js_and_get_result('decompress_async.js')
        if decompressed_data and "Decompression failed" not in decompressed_data:
            return decompressed_data
        else:
            print("Decompression failed.")
            return None
    else:
        print('binary_data invalid')
        return True

async def replayId_to_list_async(replay_id, session):
    decompressed_data = await get_gior_data_async(replay_id, session)
    if decompressed_data:
        return json.loads(decompressed_data)
    else:
        return True


In [14]:
#@title pick count/offset

replayAPICount=1
replayAPIOffset=0
#clean thsi up
try:
    replayAPICount = int(input(f"choose count(how many u wanna guess) from 1 to 200 (default: {1}): ")) #limit slider by num teams in gior, not just settings
    if not 1<=replayAPICount<=200:
      print(f"Invalid input. Using default: {1}")
      replayAPICount=1
except ValueError:
    print(f"Invalid input. Using default: {1}")
    replayAPICount=1
try:
    replayAPIOffset = int(input(f"choose offset (probably int from 0-600?) (default: {0}): ")) #limit slider by num teams in gior, not just settings
    if not 0<=replayAPIOffset<=100000:#add number of bigteam games so far
      print(f"Invalid input. Using default: {0}")
      replayAPIOffset=0
except ValueError:
    print(f"Invalid input. Using default: {0}")
    replayAPIOffset=0
#TODO: slider

choose count(how many u wanna guess) from 1 to 200 (default: 1): 3
choose offset (probably int from 0-600?) (default: 0): 1200


In [15]:
#@title get replays using ur count/offset
import requests

def fetch_replays(count=200, offset=0):
  """Fetches replays from the generals.io API with custom count and offset.

  Args:
    count: The number of replays to fetch.
    offset: The offset for fetching replays.

  Returns:
    A list of replay data.
  """
  url = f"https://generals.io/api/replays?count={count}&offset={offset}&l=bigteam"
  response = requests.get(url)
  response.raise_for_status()  # Raise an exception for bad status codes
  return response.json()

# Example usage:
# replays = fetch_replays(count=10, offset=50)
replays = fetch_replays(replayAPICount, replayAPIOffset)

import json
# print(json.dumps(replays, indent=2))

replayIds = [x['id'] for x in replays]
rankings = {}
for x in replays:
  rankings[x['id']] = [y['name'] for y in x['ranking']]
replayIds,rankings
print('fetched bigteam replays','offset',replayAPIOffset,'count',replayAPICount)

fetched bigteam replays offset 1200 count 3


In [16]:
#@title helper 2
def list_to_dict(replayId_as_list):

  ans = {}
  ans["version"] = replayId_as_list[0]
  if ans["version"]==11:
    assert len(replayId_as_list)==22
  if ans["version"]==12:
    assert len(replayId_as_list)==25
  if ans["version"]==13:
    assert len(replayId_as_list)==26
  if ans["version"]==14:
    assert len(replayId_as_list)==27
  if ans["version"]==15:
    assert len(replayId_as_list)==27
  #yandere dev coding style lol

  if ans["version"] not in [11,12,13,14,15]:
    assert False, f'what is version{ans["version"]} list length? this one is {len(replayId_as_list)}'


  ans["id"] = replayId_as_list[1]
  ans["mapWidth"] = replayId_as_list[2]
  ans["mapHeight"] = replayId_as_list[3]
  ans["usernames"] = replayId_as_list[4]
  ans["stars"] = replayId_as_list[5]
  ans["cities"] = replayId_as_list[6]
  ans["cityArmies"] = replayId_as_list[7]
  ans["generals"] = replayId_as_list[8]
  ans["mountains"] = replayId_as_list[9]
  ans["moves"] = replayId_as_list[10]
  ans["afks"] = replayId_as_list[11]
  ans["teams"] = replayId_as_list[12]
  ans["map"] = replayId_as_list[13]
  ans["neutrals"] = replayId_as_list[14]
  ans["neutralArmies"] = replayId_as_list[15]
  ans["swamps"] = replayId_as_list[16]

  ans["chat"] = replayId_as_list[17]# print(ans['chat'])
  ans["chat"]=[[message[0],message[1],ans['usernames'][message[2]],message[3]] for message in ans['chat']] if ans["version"]>=11 else ans["chat"] #[dead] and [team] tags on newer versions
  if ans["version"]<11:
    assert False, 'bruh'

  ans["playerColors"] = replayId_as_list[18]
  ans["lights"] = replayId_as_list[19]
  nested_params = replayId_as_list[20]
  ans["speed"] = nested_params[0]
  ans["city_density"] = nested_params[1]
  ans["mountain_density"] = nested_params[2]
  ans["swamp_density"] = nested_params[3]
  for i in range(4,len(nested_params)):
    ans["nested_params["+str(i)+"]"] = nested_params[i]

  ans["modifiers"] = replayId_as_list[21] #list of modifier indexes
  for i in range(22,len(replayId_as_list)):
    ans["replayId_as_list["+str(i)+"]"] = replayId_as_list[i]


  ans["replayId_as_list[22]"] = replayId_as_list[22]
  ans["replayId_as_list[23]"] = replayId_as_list[23]
  ans["replayId_as_list[24]"] = replayId_as_list[24]
  ans["replayId_as_list[25]"] = replayId_as_list[25]
  ans["pings"] = replayId_as_list[26] if len(replayId_as_list)>=27 else None

  # ans["replayId_as_list[27]"] = replayId_as_list[27]

  return ans


In [17]:
#@title fetch giors. speed's around 4 replays/second
#TODO: early filtering based on team config
import datetime
def print_current_time():
  now = datetime.datetime.now()
  print(f"Current time: {now}")
import asyncio
import aiohttp

import tqdm

collectGIORs = []

# replayIds = [x[0] for x in errorsList] for retries...



MAX_CONCURRENT_REQUESTS = [40,100,160][2]  # Adjustable window size

download_gior_async_hit_miss = [0,0] #hit/miss
semaphore = asyncio.Semaphore(MAX_CONCURRENT_REQUESTS)
replaysDoneAsync = 0
errorsList = []; success_errorFetching = [0,0]
async def fetch(session,task,len_replayInfoList_copy):
    # apiRanking,replayId,apiStarted,apiType = task
    replayId = task



    async with semaphore:
        try:


            # print('replayId',replayId)


            #raw_data = response#await response.read() #vs response.content  # Get binary data
            raw_data = await replayId_to_list_async(replayId, session)


            global replaysDoneAsync; global starttime
            replaysDoneAsync+=1
            if (replaysDoneAsync%100==0):
              print(f"{replayId},apiStarted: replays/s {round(replaysDoneAsync/((datetime.datetime.now()-starttime).total_seconds()+0.001),4)}, {replaysDoneAsync}/{len_replayInfoList_copy}")
            if (replaysDoneAsync%100==0):
              print('errorsList',errorsList)
            #print('raw_data',raw_data)



            testreplayexample = list_to_dict(raw_data)
            collectGIORs.append(testreplayexample)

            # print('raw_data',raw_data)
            success_errorFetching[0]+=1
            return raw_data

        except Exception as e:
            success_errorFetching[1]+=1
            print(f"[2]Error fetching {replayId}: {e}")
            errorsList.append([replayId,f"[2]Error fetching {replayId}: {e}"])
            return None
    print(f"{replayId}: {testreplayexample.status} done!")

async def main(replayInfoList_copy,len_replayInfoList_copy):
    async with aiohttp.ClientSession() as session:
        tasks = [fetch(session,task,len_replayInfoList_copy) for task in tqdm.tqdm(replayInfoList_copy, desc="Processing replays")]
        return await asyncio.gather(*tasks)


starttime = datetime.datetime.now()

results = await main(replayIds.copy(),len(replayIds))
if errorsList!=[]:
  print('datetime.datetime.now()-starttime',datetime.datetime.now()-starttime)
  print('len(replayIds)',len(replayIds))
  print('len(collectGIORs)',len(collectGIORs))
  print('hit/miss',download_gior_async_hit_miss)
  print('success_errorFetching',success_errorFetching)
  print('errorsList',errorsList)
  assert False, 'bad fetch'+str(errorsList)
else:
  print('fetched everything successfully')

for x in collectGIORs:
  # x["moves"]=x["moves"][:3]
  # print(json.dumps(x, indent=2))
  # print(x["id"],x["teams"])#filter out unwanted team config
  print()


Processing replays: 100%|██████████| 3/3 [00:00<00:00, 14347.68it/s]


fetched everything successfully





In [18]:
#@title creating answerkey
answerKey = []
indexToReplayId = []
for idx in range(len(collectGIORs)):
  x = collectGIORs[idx]
  testthingy = [x['teams'],x['usernames']]
  # print(x['id'],rankings[x['id']],testthingy,rankings[x['id']][0]) #testthingy[1]
  ansWinningTeam = -1
  for i in range(len(testthingy[1])):
    if (testthingy[1][i] == rankings[x['id']][0]):
      # print('winning team',testthingy[0][i])
      ansWinningTeam = testthingy[0][i]
      break
  assert ansWinningTeam!=-1
  answerKey.append(ansWinningTeam)
  indexToReplayId.append(x['id'])
# answerKey

In [19]:
#@title [disabled] font size stuff: 16px
#kinda zooms in/out when u move slider lol
# from IPython.display import HTML
# from IPython import get_ipython
# def adjust_font_size():
#     display(HTML('''<style> body { font-size: 16px; } </style>'''))
# shell = get_ipython()
# if shell:
#     shell.events.register('pre_execute', adjust_font_size)

In [20]:
#@title generate displays
indexToHex = {-1:'25;25;25', 0:'255;60;60',1:'157;189;255',2:'60;188;60',3:'60;188;188',4:'255;158;109',5:'255;110;255',
                          6:'188;60;156',7:'188;60;60',8:'236;219;108',9:'214;159;96',10:'60;60;255',11:'132;121;199',
                          12:'133;169;28',13:'248;115;117',14:'180;127;202',15:'180;153;113',
              'city':'195;195;195','mountain':'75;75;75'}; #NOTE:make all teammates same color. color doesnt match the in-game color for now
descriptions = []
for onegior in collectGIORs:

  mapString = [[" ." for _ in range(onegior['mapWidth'])] for _ in range(onegior['mapHeight'])]
  #assume no swamp/other tiles
  for index in onegior['mountains']:
    mapString[index//onegior['mapWidth']][index%onegior['mapWidth']] = '\033[;48;2;'+indexToHex['mountain']+'m'+' Δ'+'\033[0m'
  for i in range(len(onegior['cities'])):
    index = onegior['cities'][i]; armies = onegior['cityArmies'][i];
    mapString[index//onegior['mapWidth']][index%onegior['mapWidth']] = '\033[;48;2;'+indexToHex['city']+'m'+str(armies)+'\033[0m' #assume 2 digit
  for i in range(len(onegior['generals'])):
    index = onegior['generals'][i]; team = onegior['teams'][i];
    mapString[index//onegior['mapWidth']][index%onegior['mapWidth']] = '\033[;48;2;'+indexToHex[team]+'m'+'T'+str(team)+'\033[0m'
  prepareDescriptions = [onegior['id']]
  for row in mapString:
    # print(''.join(row))
    prepareDescriptions.append(''.join(row))
  descriptions.append('\n'.join(prepareDescriptions))

In [21]:
#@title make your guesses!
guesses = []
labels = ['Guess Winning Team '+longstr[:9] for longstr in descriptions] #lazy hacky getting replayid
for label, desc in zip(labels, descriptions):
    print(desc)
    guessedteamidx = 0
    try:
        guessedteamidx = int(input(f"guess winning team 0 to {CONST_SETTINGS['numTeams']-1} (default: {0}): ")) #limit slider by num teams in gior, not just settings
        if not 0<=guessedteamidx<=CONST_SETTINGS['numTeams']-1:
          print(f"Invalid input. Using default: {0}")
          guessedteamidx=0
    except ValueError:
        print(f"Invalid input. Using default: {0}")
        guessedteamidx=0

    print(guessedteamidx)
    guesses.append(guessedteamidx)

#TODO: slider

-8rkgcQqm
45 . . . . Δ . . . Δ . . . . . Δ . . .47 . . . . Δ . . . . . .
 . . Δ . . Δ . Δ . . . . . . . . . . . . . . . . . . . ΔT1 . .
 . Δ45 Δ . Δ . . . . Δ Δ . .47 . Δ . . .T1 . Δ . . Δ . . . . .
 . . . . . . . . . . . Δ Δ . . . . . . Δ . . . Δ . Δ . . Δ . .
 .40 . . . Δ . Δ . Δ . Δ Δ . Δ . . Δ . . .T1 Δ Δ . . . . . . .
 . . . . . . Δ . . . . . . . . . . . . Δ . . . . . Δ . . . Δ .
 . . . . Δ . Δ . . . . . . . . . . . . . . . . . . Δ . . . . .
 . . . . . . Δ . Δ Δ Δ . .42 . . Δ . . Δ . . . . . . .T1 . . Δ
 . . . . Δ . . Δ41 Δ . . Δ . . . .48 . . Δ . . . . . .40 . Δ .
 . . . . Δ . . . . . . . . . . . . .43 . . . Δ . Δ Δ . Δ . . .
 Δ . . .T0 . . . . . . . . . . Δ Δ . .49 . Δ . . . . Δ Δ . . .
 . . . . . . . . . . . . . . . . . . Δ . . . . . . Δ . Δ Δ . Δ
 Δ Δ . . Δ . . . . Δ . . Δ Δ . . . . . . . . . Δ . .47 . . . .
 .46 . .T0 . . . .T0 . . Δ . . . . . . . . . . Δ Δ . . . . . Δ
 . . . . . Δ . . . . . . . Δ Δ . . . Δ . Δ . . . Δ . Δ . Δ . .
 . Δ47 . . . Δ . . Δ . . . . . . Δ . . . Δ . 

In [22]:
#@title show results!
# guesses = [slider.value for slider in sliders]#slider fails lol
# print(indexToReplayId)
assert len(answerKey)==len(guesses)
yourscores=[]
for i in range(len(guesses)):
  print(descriptions[i])
  print('replayid',indexToReplayId[i],'yourguess',guesses[i],'answer',answerKey[i],'correct?',answerKey[i]==guesses[i])
  yourscores.append(answerKey[i]==guesses[i])
print('score',sum(yourscores)/len(yourscores)*100,'%')

-8rkgcQqm
45 . . . . Δ . . . Δ . . . . . Δ . . .47 . . . . Δ . . . . . .
 . . Δ . . Δ . Δ . . . . . . . . . . . . . . . . . . . ΔT1 . .
 . Δ45 Δ . Δ . . . . Δ Δ . .47 . Δ . . .T1 . Δ . . Δ . . . . .
 . . . . . . . . . . . Δ Δ . . . . . . Δ . . . Δ . Δ . . Δ . .
 .40 . . . Δ . Δ . Δ . Δ Δ . Δ . . Δ . . .T1 Δ Δ . . . . . . .
 . . . . . . Δ . . . . . . . . . . . . Δ . . . . . Δ . . . Δ .
 . . . . Δ . Δ . . . . . . . . . . . . . . . . . . Δ . . . . .
 . . . . . . Δ . Δ Δ Δ . .42 . . Δ . . Δ . . . . . . .T1 . . Δ
 . . . . Δ . . Δ41 Δ . . Δ . . . .48 . . Δ . . . . . .40 . Δ .
 . . . . Δ . . . . . . . . . . . . .43 . . . Δ . Δ Δ . Δ . . .
 Δ . . .T0 . . . . . . . . . . Δ Δ . .49 . Δ . . . . Δ Δ . . .
 . . . . . . . . . . . . . . . . . . Δ . . . . . . Δ . Δ Δ . Δ
 Δ Δ . . Δ . . . . Δ . . Δ Δ . . . . . . . . . Δ . .47 . . . .
 .46 . .T0 . . . .T0 . . Δ . . . . . . . . . . Δ Δ . . . . . Δ
 . . . . . Δ . . . . . . . Δ Δ . . . Δ . Δ . . . Δ . Δ . Δ . .
 . Δ47 . . . Δ . . Δ . . . . . . Δ . . . Δ . 

In [23]:
print('score',sum(yourscores)/len(yourscores)*100,'%')
print('thanks for playing! what things influenced ur decision? any suggestions? add feedback here https://discord.com/channels/252596486628573185/1394108615455019168/1394108615455019168')

score 66.66666666666666 %
thanks for playing! what things influenced ur decision? any suggestions? add feedback here https://discord.com/channels/252596486628573185/1394108615455019168/1394108615455019168
